# Session 8: Capstone Project + Deployment
**Duration:** 1.5 Hours | **Day 2** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Integrate RAG, agents, and MCP into a complete application
2. Build a Streamlit-based UI for an AI assistant
3. Deploy and test the application locally
4. Understand production deployment considerations

---

## Prerequisites
- Completed all previous sessions
- Ollama running with `llama3.2` and `nomic-embed-text`
- Install: `pip install streamlit langchain langchain-ollama faiss-cpu`

---
## Part 1: Capstone Options (5 min)

Choose ONE capstone project to build:

| Option | Description | Components |
|--------|-------------|------------|
| A | College Info Assistant | RAG (handbook) + Tools (fees, GPA) + Chat UI |
| B | AI Career Mentor Bot | RAG (career guides) + Agent (analysis) + Chat UI |
| C | Interview Prep Coach | Multi-Agent (Q generator + evaluator) + Chat UI |
| D | Research Paper Summarizer | RAG (papers) + Multi-Agent (summarize + critique) |

We will build **Option A: College Info Assistant** together, as it ties all workshop concepts.

### Architecture

```
User (Streamlit UI)
        |
        v
  [Chat Interface]
        |
        v
  [Router Agent] -- decides the approach
      |          |
      v          v
  [RAG Chain]  [Tool Agent]
      |          |
      v          v
  FAISS DB    Calculator/
 (handbook)   GPA/Fees
      |          |
      +----+-----+
           |
           v
     [Response]
```

---
## Part 2: Build the Backend (Hands-On - 30 min)

### 2.1 Create the RAG + Agent backend

In [ ]:
# 2.1 Setup and Imports

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
import json

# Initialize models
llm = ChatOllama(model="llama3.2", temperature=0.3)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Models initialized!")

In [ ]:
# 2.2 Create/Load Vector Store

# Sample college data (in production, load from files)
college_docs = [
    "SIT Tumkur offers B.E. programs in Computer Science, Information Science, Electronics, Mechanical, and Civil Engineering.",
    "Admission is based on CET rank for management quota and COMEDK for other seats. Direct admission available for NRI quota.",
    "Tuition fee for B.E. Computer Science is approximately Rs. 1,50,000 per year. Hostel fee is Rs. 80,000 per year.",
    "The campus has a central library with over 50,000 books, e-journals access, and a digital library section.",
    "Placement cell has tie-ups with 200+ companies including TCS, Infosys, Wipro, and Accenture.",
    "Average placement package is Rs. 4.5 LPA. Highest package recorded is Rs. 18 LPA.",
    "Students must maintain minimum 85% attendance to be eligible for examinations.",
    "Each semester has internal assessment (50 marks) and external examination (100 marks) for each subject.",
    "The college has an incubation center supporting student startups with mentorship and seed funding.",
    "Sports facilities include cricket ground, basketball court, volleyball court, indoor games, and a gymnasium.",
    "Technical fests and cultural events are organized annually. Major events include TechFiesta and Sanskriti.",
    "Anti-ragging committee is active with zero tolerance policy. Helpline: 1800-180-5522.",
    "Hostel has separate blocks for boys and girls. Wi-Fi enabled with 24/7 security.",
    "The training and placement cell conducts aptitude training, mock interviews, and soft skills workshops.",
    "CGPA to percentage conversion: Percentage = (CGPA - 0.75) * 10. Minimum CGPA for distinction is 7.5."
]

# Create vector store
from langchain_core.documents import Document
docs = [Document(page_content=text, metadata={"source": "college_info"}) for text in college_docs]

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vector store created with {len(college_docs)} documents.")

In [ ]:
# 2.3 Define Tools

def calculate_fee(program: str, years: int = 4, include_hostel: bool = True) -> str:
    """Calculate total fees for a program."""
    fee_data = {
        "cse": 150000,
        "ise": 140000,
        "ece": 130000,
        "me": 120000,
        "ce": 110000
    }
    program_lower = program.lower()
    tuition = fee_data.get(program_lower, 130000)
    hostel = 80000 if include_hostel else 0
    total = (tuition + hostel) * years
    return f"Program: {program.upper()}, Duration: {years} years, Tuition: Rs.{tuition:,}/year, Hostel: Rs.{hostel:,}/year, Total: Rs.{total:,}"

def calculate_gpa(marks: list) -> str:
    """Calculate GPA from a list of marks."""
    if not marks:
        return "No marks provided."
    grade_points = []
    for m in marks:
        if m >= 90: grade_points.append(10)
        elif m >= 80: grade_points.append(9)
        elif m >= 70: grade_points.append(8)
        elif m >= 60: grade_points.append(7)
        elif m >= 50: grade_points.append(6)
        elif m >= 40: grade_points.append(5)
        else: grade_points.append(0)
    gpa = sum(grade_points) / len(grade_points)
    percentage = (gpa - 0.75) * 10
    return f"Marks: {marks}, Grade Points: {grade_points}, GPA: {gpa:.2f}, Approx Percentage: {percentage:.1f}%"

def get_placement_stats(branch: str = "") -> str:
    """Get placement statistics."""
    stats = {
        "cse": {"placed": 92, "avg": 5.2, "highest": 18, "companies": ["TCS", "Infosys", "Wipro", "Amazon"]},
        "ise": {"placed": 88, "avg": 4.8, "highest": 15, "companies": ["TCS", "Cognizant", "Capgemini"]},
        "ece": {"placed": 75, "avg": 4.0, "highest": 12, "companies": ["Bosch", "Texas Instruments", "Infosys"]},
        "me": {"placed": 65, "avg": 3.5, "highest": 8, "companies": ["Toyota", "L&T", "Tata Motors"]},
    }
    branch_lower = branch.lower() if branch else ""
    if branch_lower in stats:
        s = stats[branch_lower]
        return f"Branch: {branch.upper()}, Placed: {s['placed']}%, Avg Package: Rs.{s['avg']} LPA, Highest: Rs.{s['highest']} LPA, Top Companies: {', '.join(s['companies'])}"
    elif branch:
        return f"No specific stats for {branch}. Overall: 78% placed, Avg Rs.4.5 LPA, Highest Rs.18 LPA."
    else:
        return "Overall: 78% students placed, Average Rs.4.5 LPA, Highest Rs.18 LPA, 200+ recruiting companies."

tools_registry = {
    "calculate_fee": calculate_fee,
    "calculate_gpa": calculate_gpa,
    "get_placement_stats": get_placement_stats
}

print("Tools defined: calculate_fee, calculate_gpa, get_placement_stats")

In [ ]:
# 2.4 Build the College Assistant Backend

class CollegeAssistant:
    """Complete college assistant combining RAG, tools, and agent logic."""

    def __init__(self, llm, retriever, tools: dict):
        self.llm = llm
        self.retriever = retriever
        self.tools = tools
        self.chat_history = []

    def _classify_query(self, query: str) -> str:
        """Determine if query needs RAG, tools, or direct answer."""
        classification_prompt = f"""Classify this query into one category:
- "rag": if it's about college info, policies, facilities, courses
- "fee_calc": if it asks to calculate fees
- "gpa_calc": if it asks to calculate GPA or grade points
- "placement": if it asks about placement statistics
- "general": for greetings, general questions, or unrelated topics

Query: {query}

Return ONLY the category word, nothing else."""

        response = self.llm.invoke(classification_prompt)
        category = response.content.strip().lower().strip('"').strip("'")

        # Normalize
        if "fee" in category: return "fee_calc"
        if "gpa" in category or "grade" in category: return "gpa_calc"
        if "placement" in category: return "placement"
        if "rag" in category: return "rag"
        return "general"

    def _handle_rag(self, query: str) -> str:
        """Handle RAG queries."""
        docs = self.retriever.invoke(query)
        context = "\n".join([doc.page_content for doc in docs])

        rag_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a helpful college information assistant for SIT Tumkur.
Answer based on the provided context. If the context does not contain the answer, say so.
Be concise and helpful.

Context:
{context}"""),
            ("human", "{question}")
        ])

        chain = rag_prompt | self.llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

    def _handle_fee(self, query: str) -> str:
        """Handle fee calculation queries."""
        # Extract program from query
        extract_prompt = f"""Extract the program/branch name from this query. Return just the abbreviation.
CSE = Computer Science, ISE = Information Science, ECE = Electronics, ME = Mechanical, CE = Civil
Query: {query}
Return ONLY the abbreviation (e.g., CSE). If unclear, return CSE."""
        program = self.llm.invoke(extract_prompt).content.strip().upper()
        if program not in ["CSE", "ISE", "ECE", "ME", "CE"]:
            program = "CSE"

        fee_result = self.tools["calculate_fee"](program)

        # Generate natural response
        response_prompt = f"""Given this fee calculation result, provide a helpful response to the student.
Result: {fee_result}
Original question: {query}
Be friendly and include all the numbers."""
        return self.llm.invoke(response_prompt).content

    def _handle_gpa(self, query: str) -> str:
        """Handle GPA calculation queries."""
        # Use example marks if none provided
        import re
        numbers = re.findall(r'\b(\d{2,3})\b', query)
        marks = [int(n) for n in numbers if 0 <= int(n) <= 100]

        if not marks:
            return "Please provide your marks (out of 100) for each subject. Example: 'Calculate GPA for marks 85, 78, 92, 65, 70'"

        gpa_result = self.tools["calculate_gpa"](marks)

        response_prompt = f"""Given this GPA calculation, provide a helpful response.
Result: {gpa_result}
Original question: {query}
Include the GPA and percentage. Be encouraging."""
        return self.llm.invoke(response_prompt).content

    def _handle_placement(self, query: str) -> str:
        """Handle placement queries."""
        # Extract branch
        extract_prompt = f"""Extract the branch name from this query about placements.
Options: CSE, ISE, ECE, ME, CE, or OVERALL if no specific branch.
Query: {query}
Return ONLY the abbreviation."""
        branch = self.llm.invoke(extract_prompt).content.strip().upper()
        if branch == "OVERALL":
            branch = ""

        stats = self.tools["get_placement_stats"](branch)

        response_prompt = f"""Given these placement statistics, provide a detailed response.
Stats: {stats}
Question: {query}
Be informative and encouraging."""
        return self.llm.invoke(response_prompt).content

    def chat(self, query: str) -> str:
        """Main chat interface."""
        # Classify the query
        category = self._classify_query(query)

        # Route to appropriate handler
        handlers = {
            "rag": self._handle_rag,
            "fee_calc": self._handle_fee,
            "gpa_calc": self._handle_gpa,
            "placement": self._handle_placement,
            "general": lambda q: self.llm.invoke(
                f"You are a helpful college assistant for SIT Tumkur. Respond to: {q}"
            ).content
        }

        handler = handlers.get(category, handlers["general"])
        response = handler(query)

        # Store in chat history
        self.chat_history.append({"role": "user", "content": query, "category": category})
        self.chat_history.append({"role": "assistant", "content": response})

        return response

# Create the assistant
assistant = CollegeAssistant(llm=llm, retriever=retriever, tools=tools_registry)
print("College Assistant initialized!")

In [ ]:
# 2.5 Test the backend

test_queries = [
    "What is the minimum attendance required?",
    "Calculate total fees for CSE with hostel",
    "Calculate my GPA for marks 85 78 92 65 70",
    "What are the placement stats for CSE?",
    "Tell me about the sports facilities"
]

for query in test_queries:
    print(f"\nQ: {query}")
    response = assistant.chat(query)
    print(f"A: {response[:200]}..." if len(response) > 200 else f"A: {response}")
    print("-" * 50)

---
## Part 3: Build the Streamlit UI (Hands-On - 30 min)

Create a web application using Streamlit.

In [ ]:
# 3.1 Generate the Streamlit app file

streamlit_code = '''
import streamlit as st
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import re

# --- Page Config ---
st.set_page_config(
    page_title="SIT Tumkur - College Assistant",
    page_icon="🎓",
    layout="centered"
)

# --- Custom CSS ---
st.markdown("""
<style>
.stApp {
    max-width: 800px;
    margin: 0 auto;
}
.chat-message {
    padding: 1rem;
    border-radius: 0.5rem;
    margin-bottom: 0.5rem;
}
</style>
""", unsafe_allow_html=True)

# --- Initialize Models (cached) ---
@st.cache_resource
def init_llm():
    return ChatOllama(model="llama3.2", temperature=0.3)

@st.cache_resource
def init_vectorstore():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    college_docs = [
        "SIT Tumkur offers B.E. programs in Computer Science, Information Science, Electronics, Mechanical, and Civil Engineering.",
        "Admission is based on CET rank for management quota and COMEDK for other seats.",
        "Tuition fee for B.E. Computer Science is approximately Rs. 1,50,000 per year. Hostel fee is Rs. 80,000 per year.",
        "The campus has a central library with over 50,000 books and e-journals access.",
        "Placement cell has tie-ups with 200+ companies including TCS, Infosys, Wipro, and Accenture.",
        "Average placement package is Rs. 4.5 LPA. Highest package recorded is Rs. 18 LPA.",
        "Students must maintain minimum 85% attendance to be eligible for examinations.",
        "Each semester has internal assessment (50 marks) and external examination (100 marks).",
        "The college has an incubation center supporting student startups.",
        "Sports facilities include cricket ground, basketball court, volleyball court, and gymnasium.",
        "Anti-ragging committee is active with zero tolerance policy. Helpline: 1800-180-5522.",
        "Hostel has separate blocks for boys and girls. Wi-Fi enabled with 24/7 security.",
        "Training and placement cell conducts aptitude training and mock interviews.",
        "CGPA to percentage conversion: Percentage = (CGPA - 0.75) * 10."
    ]
    docs = [Document(page_content=text) for text in college_docs]
    vectorstore = FAISS.from_documents(docs, embeddings)
    return vectorstore

# --- Tool Functions ---
def calculate_fee(program, years=4, include_hostel=True):
    fee_data = {"cse": 150000, "ise": 140000, "ece": 130000, "me": 120000, "ce": 110000}
    tuition = fee_data.get(program.lower(), 130000)
    hostel = 80000 if include_hostel else 0
    total = (tuition + hostel) * years
    return f"Program: {program.upper()}, Duration: {years} years, Tuition: Rs.{tuition:,}/year, Hostel: Rs.{hostel:,}/year, Total: Rs.{total:,}"

def calculate_gpa(marks):
    grade_points = []
    for m in marks:
        if m >= 90: grade_points.append(10)
        elif m >= 80: grade_points.append(9)
        elif m >= 70: grade_points.append(8)
        elif m >= 60: grade_points.append(7)
        elif m >= 50: grade_points.append(6)
        elif m >= 40: grade_points.append(5)
        else: grade_points.append(0)
    gpa = sum(grade_points) / len(grade_points)
    pct = (gpa - 0.75) * 10
    return f"Marks: {marks}, GPA: {gpa:.2f}, Percentage: {pct:.1f}%"

def get_placement_stats(branch=""):
    stats = {
        "cse": {"placed": 92, "avg": 5.2, "highest": 18},
        "ise": {"placed": 88, "avg": 4.8, "highest": 15},
        "ece": {"placed": 75, "avg": 4.0, "highest": 12},
        "me": {"placed": 65, "avg": 3.5, "highest": 8},
    }
    b = branch.lower()
    if b in stats:
        s = stats[b]
        return f"Branch: {branch.upper()}, Placed: {s[\"placed\"]}%, Avg: Rs.{s[\"avg\"]} LPA, Highest: Rs.{s[\"highest\"]} LPA"
    return "Overall: 78% placed, Avg Rs.4.5 LPA, Highest Rs.18 LPA"

# --- Assistant Logic ---
def classify_query(llm, query):
    prompt = f"""Classify this query into one category:
- "rag": college info, policies, facilities
- "fee_calc": calculate fees
- "gpa_calc": calculate GPA
- "placement": placement statistics
- "general": greetings, other
Query: {query}
Return ONLY the category word."""
    response = llm.invoke(prompt).content.strip().lower()
    if "fee" in response: return "fee_calc"
    if "gpa" in response: return "gpa_calc"
    if "placement" in response: return "placement"
    if "rag" in response: return "rag"
    return "general"

def get_response(llm, vectorstore, query):
    category = classify_query(llm, query)

    if category == "rag":
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        docs = retriever.invoke(query)
        context = "\\n".join([d.page_content for d in docs])
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a college assistant for SIT Tumkur. Answer based on context. Context: {context}"),
            ("human", "{question}")
        ])
        chain = prompt | llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

    elif category == "fee_calc":
        extract = llm.invoke(f"Extract branch abbreviation (CSE/ISE/ECE/ME/CE) from: {query}. Return ONLY abbreviation.").content.strip()
        if extract.upper() not in ["CSE","ISE","ECE","ME","CE"]: extract = "CSE"
        result = calculate_fee(extract)
        return llm.invoke(f"Respond helpfully about fees. Data: {result}. Question: {query}").content

    elif category == "gpa_calc":
        numbers = re.findall(r"\\b(\\d{{2,3}})\\b", query)
        marks = [int(n) for n in numbers if 0 <= int(n) <= 100]
        if not marks:
            return "Please provide marks (out of 100). Example: Calculate GPA for 85, 78, 92, 65, 70"
        result = calculate_gpa(marks)
        return llm.invoke(f"Respond about GPA. Data: {result}. Question: {query}").content

    elif category == "placement":
        extract = llm.invoke(f"Extract branch (CSE/ISE/ECE/ME or OVERALL) from: {query}. Return abbreviation.").content.strip()
        if extract.upper() == "OVERALL": extract = ""
        result = get_placement_stats(extract)
        return llm.invoke(f"Respond about placements. Data: {result}. Question: {query}").content

    else:
        return llm.invoke(f"You are SIT Tumkur college assistant. Respond to: {query}").content

# --- Main UI ---
st.title("SIT Tumkur - College Assistant")
st.caption("Ask about admissions, fees, placements, GPA calculation, and more.")

# Initialize
llm = init_llm()
vectorstore = init_vectorstore()

# Chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Welcome! I am the SIT Tumkur College Assistant. Ask me about admissions, fees, placements, GPA calculation, or any college-related queries."}
    ]

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if prompt := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = get_response(llm, vectorstore, prompt)
        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

# Sidebar
with st.sidebar:
    st.header("Quick Actions")
    if st.button("Fee Calculator"):
        st.session_state.sidebar_query = "Calculate total fees for CSE with hostel"
    if st.button("Placement Stats"):
        st.session_state.sidebar_query = "What are the placement statistics for CSE?"
    if st.button("GPA Calculator"):
        st.session_state.sidebar_query = "How to calculate GPA?"

    st.divider()
    st.header("About")
    st.write("Built with LangChain, Ollama, FAISS, and Streamlit.")
    st.write("Powered by Llama 3.2 running locally.")

    if st.button("Clear Chat"):
        st.session_state.messages = [
            {"role": "assistant", "content": "Chat cleared. How can I help you?"}
        ]
        st.rerun()
'''

# Save to file
with open("../data/college_assistant_app.py", "w") as f:
    f.write(streamlit_code)

print("Streamlit app saved to: ../data/college_assistant_app.py")
print("\nTo run: streamlit run data/college_assistant_app.py")

---
## Part 4: Running and Testing the App (10 min)

### 4.1 Running Locally

```bash
# Make sure Ollama is running
ollama serve

# In another terminal, navigate to the workshop directory
cd workshop

# Run the Streamlit app
streamlit run data/college_assistant_app.py
```

The app will open at `http://localhost:8501`

### 4.2 Test Scenarios

Try these queries in the app:

| Query Type | Example |
|------------|----------|
| RAG | "What are the library facilities?" |
| Fee Calc | "Calculate CSE fees for 4 years with hostel" |
| GPA | "Calculate GPA for marks 85, 78, 92, 65, 70" |
| Placement | "Placement stats for CSE branch" |
| General | "Hello, what can you do?" |

---
## Part 5: Production Considerations (Theory - 10 min)

### 5.1 Deployment Architecture for Production

```
Internet
    |
    v
[Reverse Proxy (Nginx)]
    |
    v
[Streamlit App (Docker Container)]
    |
    +---> [Ollama Server (GPU Machine)]
    +---> [Vector DB (FAISS / Chroma / Pinecone)]
    +---> [MCP Servers (Docker Containers)]
```

### 5.2 Checklist for Production

| Area | Consideration |
|------|---------------|
| **Security** | Input sanitization, rate limiting, authentication |
| **Scalability** | Load balancing, GPU scaling for LLM inference |
| **Data** | Regular vector DB updates, document versioning |
| **Monitoring** | Logging all queries, response times, error rates |
| **Cost** | Token usage tracking, caching frequent queries |
| **Quality** | Human feedback loop, A/B testing prompts |
| **Privacy** | PII detection, data retention policies |

In [ ]:
# 5.3 Docker Deployment Setup

dockerfile_content = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health

ENTRYPOINT ["streamlit", "run", "college_assistant_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
"""

requirements_content = """streamlit>=1.30.0
langchain>=0.1.0
langchain-ollama>=0.1.0
langchain-community>=0.0.10
faiss-cpu>=1.7.4
requests>=2.31.0
"""

compose_content = """version: '3.8'

services:
  ollama:
    image: ollama/ollama:latest
    ports:
      - "11434:11434"
    volumes:
      - ollama_data:/root/.ollama
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]

  college-assistant:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8501:8501"
    environment:
      - OLLAMA_HOST=http://ollama:11434
    depends_on:
      - ollama

volumes:
  ollama_data:
"""

# Save deployment files
with open("../data/Dockerfile.app", "w") as f:
    f.write(dockerfile_content)

with open("../data/requirements.txt", "w") as f:
    f.write(requirements_content)

with open("../data/docker-compose-app.yml", "w") as f:
    f.write(compose_content)

print("Deployment files created:")
print("  - Dockerfile.app")
print("  - requirements.txt")
print("  - docker-compose-app.yml")
print("\nTo deploy:")
print("  docker compose -f data/docker-compose-app.yml up --build")

---
## Part 6: Workshop Wrap-Up (5 min)

### What We Built Over 2 Days

| Session | Topic | Key Skills |
|---------|-------|------------|
| 1 | LLM Foundations | Ollama, prompt engineering, chains |
| 2 | Open-Source LLMs | Embeddings, tokenization, semantic search |
| 3 | RAG Pipeline | Document loading, chunking, FAISS, retrieval chains |
| 4 | MCP Introduction | MCP servers, tools, resources, transports |
| 5 | Agentic AI | ReAct pattern, tool use, agent loops |
| 6 | MCP Deep Dive | MCP clients, LangChain integration, Docker deployment |
| 7 | Multi-Agent Systems | Pipelines, iterative refinement, orchestration |
| 8 | Capstone + Deploy | Streamlit UI, full app, Docker deployment |

### Next Steps for Participants

1. **Explore more models**: Try `mistral`, `codellama`, `phi3` with Ollama
2. **Scale your RAG**: Use Chroma or Pinecone for larger datasets
3. **Build MCP servers**: Create tools for your domain (college ERP, library, etc.)
4. **Try frameworks**: Explore CrewAI, AutoGen, or LangGraph for advanced agents
5. **Deploy to cloud**: AWS, GCP, or Azure with GPU instances for Ollama

### Resources

- LangChain Docs: https://python.langchain.com
- Ollama: https://ollama.ai
- MCP Specification: https://modelcontextprotocol.io
- Streamlit: https://streamlit.io
- FAISS: https://github.com/facebookresearch/faiss

---

### Thank You!
Review the notebooks, experiment with the code, and build something of your own.